# Phase 5: Targeted Analysis 9: Information Flow (Algorithm Preservation Test)

## Overview

NB01 showed the universal core is partially sufficient, NB07 tests necessity. But even if the core is both sufficient and necessary, we don't know if it implements the **same algorithm** as the full circuit. This notebook tests algorithm preservation by comparing P(correct) trajectories (logit lens) across layers.

If the universal core's P(correct) trajectory has the same **shape** as the full circuit's trajectory (high Pearson r) but at a lower **amplitude** (slope < 1), then the core runs the same computation: just weaker.

## Key Questions

1. Does the universal core produce the same per-layer P(correct) trajectory shape as the full circuit?
2. Is the convergence layer preserved (same layer where output "crystallizes")?
3. Is the amplitude ratio consistent across bands?

## Hypotheses

- H1: Trajectory correlation r > 0.95 (same shape)
- H2: Convergence layer shift < 1 layer
- H3: Amplitude ratio < 1 but consistent across bands (same algorithm, attenuated)

## Method

Uses Phase 3's source-level mean ablation approach with TransformerLens:
1. Load model with 'fold_ln=True' (matching circuit discovery)
2. Build mean activation cache over all bands x draws
3. Apply source-level ablation for universal core edges
4. Capture 'hook_resid_post' at prediction position for each layer
5. Apply unembedding: P(correct | layer) = softmax(resid @ W_U)[target_id]

## Sections

1. Setup & Imports
2. Build Universal Core Prune Scores
3. Extract Universal Core Logit Lens (GPU)
4. Compute Convergence Layers
5. Trajectory Shape Comparison
6. Visualizations
7. Summary

## Data Sources

- 60 circuits: 'circuit_discovery/circuits/{model}/{band}/{draw}/prune_scores.pkl'
- Test data: 'LSC_data/datasets/matched/{draw}/{band}/test.json'
- Phase 3 trajectories: '03_Phase_Representational/outputs/logit_lens/circuit/analysis/03c_circuit_prob_correct.csv'
- Phase 3 convergence: '03_Phase_Representational/outputs/logit_lens/circuit/analysis/03c_circuit_convergence.csv'
- Phase 3 base trajectories: '03_Phase_Representational/outputs/logit_lens/base/analysis/03_prob_correct_trajectory.csv'

**GPU Required**: YES (~90 min, TransformerLens forward passes with hooks)

In [1]:
import os
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import pickle
import gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from scipy import stats as sp_stats
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Paths
ANALYSIS_ROOT = Path("LSC_circuit_analysis")
ISC_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
LSC_DIR = ISC_ROOT / "LSC_circuits"
CIRCUITS_DIR = LSC_DIR / "circuit_discovery" / "circuits"
DATA_DIR = ISC_ROOT / "LSC_data"

PHASE3_DIR = ANALYSIS_ROOT / "03_Phase_Representational"
PHASE5_DIR = ANALYSIS_ROOT / "05_Phase_Targeted"
OUT_ANALYSIS = PHASE5_DIR / "outputs" / "analysis"
OUT_VIZ = PHASE5_DIR / "outputs" / "viz"
OUT_ANALYSIS.mkdir(parents=True, exist_ok=True)
OUT_VIZ.mkdir(parents=True, exist_ok=True)

# Import Phase 3 utilities for ablation hooks and mean cache building
sys.path.insert(0, str(PHASE3_DIR))
from utils.circuit_extraction import create_ablation_hooks, load_model_circuit_mode
from utils.circuit_loading import build_mean_cache, load_prune_scores
from utils.constants import (
    MODEL_DIR_NAMES,
    MODEL_PREDICTION_POS,
    SEQ_LEN,
    MODEL_LAYERS,
    MODEL_HEADS,
    CONVERGENCE_THRESHOLD,
    DATASETS_BASE,
    HF_MODEL_NAMES,
)

# Constants
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
DRAWS = ["draw_1", "draw_2", "draw_3"]
PRED_POS = MODEL_PREDICTION_POS  # 21 (after BOS)

sns.set_theme(style="whitegrid", font_scale=1.1)


def save_figure(fig, filename):
    path = OUT_VIZ / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


print("Setup complete.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Prediction position (with BOS): {PRED_POS}")

Setup complete.
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
Prediction position (with BOS): 21


In [2]:
# Load Phase 3 reference trajectories
df_circuit_traj = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "logit_lens"
    / "circuit"
    / "analysis"
    / "03c_circuit_prob_correct.csv"
)
df_circuit_conv = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "logit_lens"
    / "circuit"
    / "analysis"
    / "03c_circuit_convergence.csv"
)
df_base_traj = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "logit_lens"
    / "base"
    / "analysis"
    / "03_prob_correct_trajectory.csv"
)

print(f"Full circuit trajectories: {len(df_circuit_traj)} rows")
print(f"  Columns: {list(df_circuit_traj.columns)}")
print(f"Full circuit convergence: {len(df_circuit_conv)} rows")
print(f"Base model trajectories: {len(df_base_traj)} rows")
print(f"  Columns: {list(df_base_traj.columns)}")

Full circuit trajectories: 1230 rows
  Columns: ['model', 'draw', 'band', 'layer', 'mean_prob_correct', 'std_prob_correct', 'median_prob_correct']
Full circuit convergence: 75 rows
Base model trajectories: 1230 rows
  Columns: ['model', 'draw', 'band', 'layer', 'mean_prob_correct', 'std_prob_correct', 'median_prob_correct']


## 2. Build Universal Core Prune Scores

Same construction as NB01/NB07: AND of all 5 bands' inf masks.

In [3]:
universal_cache = {}  # (model, draw) -> prune_scores dict

for model_name in MODELS:
    m_safe = MODEL_DIR_NAMES.get(model_name, model_name.replace("-", "_"))
    print(f"\n{model_name}:")

    for draw in DRAWS:
        all_band_scores = {}
        missing = False
        for band in BANDS:
            path = CIRCUITS_DIR / m_safe / band / draw / "prune_scores.pkl"
            if not path.exists():
                print(f"  WARNING: missing {path}")
                missing = True
                break
            with open(path, "rb") as f:
                all_band_scores[band] = pickle.load(f)
        if missing:
            continue

        # AND of all bands' inf masks
        first_band = BANDS[0]
        universal = {}
        for mod_name in all_band_scores[first_band]:
            mask = torch.ones_like(
                all_band_scores[first_band][mod_name], dtype=torch.bool
            )
            for band in BANDS:
                mask &= torch.isinf(all_band_scores[band][mod_name])
            tensor = torch.zeros_like(all_band_scores[first_band][mod_name])
            tensor[mask] = float("inf")
            universal[mod_name] = tensor

        n_edges = sum(torch.isinf(s).sum().item() for s in universal.values())
        universal_cache[(model_name, draw)] = universal
        print(f"  {draw}: {n_edges} universal edges")

print(f"\nBuilt universal prune scores for {len(universal_cache)} configurations.")


pythia-70m:


  draw_1: 297 universal edges
  draw_2: 297 universal edges
  draw_3: 311 universal edges

pythia-160m:
  draw_1: 694 universal edges
  draw_2: 723 universal edges


  draw_3: 723 universal edges

pythia-410m:


  draw_1: 1283 universal edges
  draw_2: 1301 universal edges
  draw_3: 1324 universal edges

pythia-1b:
  draw_1: 348 universal edges


  draw_2: 356 universal edges
  draw_3: 364 universal edges

pythia-1.4b:


  draw_1: 625 universal edges
  draw_2: 616 universal edges
  draw_3: 619 universal edges

Built universal prune scores for 15 configurations.


## 3. Extract Universal Core Logit Lens (GPU)

For each model:
1. Load TransformerLens model with 'fold_ln=True'
2. Build mean activation cache over ALL bands x draws
3. For each draw x band: apply universal core ablation hooks, capture residual stream, compute P(correct) per layer

In [4]:
def load_test_dataset(draw, band):
    """Load test dataset as numpy arrays matching Phase 3 format."""
    test_path = DATASETS_BASE / draw / band / "test.json"
    with open(test_path) as f:
        data = json.load(f)

    examples = data["examples"]
    input_ids = np.array([ex["token_ids"] for ex in examples], dtype=np.int64)
    target_ids = np.array([ex["target_token_id"] for ex in examples], dtype=np.int64)

    return {
        "input_ids": input_ids,
        "target_ids": target_ids,
        "n_examples": len(examples),
    }


def compute_logit_lens_from_resid(model, resid_post, target_ids):
    """Compute P(correct) at each layer from residual stream activations.

    Args:
        model: TransformerLens HookedTransformer
        resid_post: (n_examples, n_layers, d_model) numpy array
        target_ids: (n_examples,) numpy array of target token ids

    Returns:
        prob_correct: (n_examples, n_layers) numpy array of P(correct | layer)
    """
    device = next(model.parameters()).device
    n_examples, n_layers, d_model = resid_post.shape

    prob_correct = np.zeros((n_examples, n_layers), dtype=np.float32)

    with torch.no_grad():
        # W_U: (d_model, d_vocab)
        W_U = model.W_U  # Already on device
        b_U = model.b_U if hasattr(model, "b_U") and model.b_U is not None else None

        # Process in batches to avoid OOM
        batch_size = 64
        for start in range(0, n_examples, batch_size):
            end = min(start + batch_size, n_examples)

            resid_batch = torch.tensor(
                resid_post[start:end], dtype=torch.float32, device=device
            )  # (batch, n_layers, d_model)
            targets = torch.tensor(
                target_ids[start:end], dtype=torch.long, device=device
            )  # (batch,)

            for layer in range(n_layers):
                # Unembed: (batch, d_model) @ (d_model, d_vocab) = (batch, d_vocab)
                logits = resid_batch[:, layer, :] @ W_U
                if b_U is not None:
                    logits = logits + b_U

                probs = torch.softmax(logits, dim=-1)
                # P(correct) for each example
                p_correct = probs[torch.arange(len(targets)), targets]
                prob_correct[start:end, layer] = p_correct.cpu().numpy()

            del resid_batch, targets

    return prob_correct


print("Helper functions defined.")

Helper functions defined.


In [5]:
device = "cuda:0"
trajectory_rows = []
eval_idx = 0
total_evals = len(MODELS) * len(DRAWS) * len(BANDS)
BATCH_SIZE = 64

for model_name in MODELS:
    n_layers = MODEL_LAYERS[model_name]
    n_heads = MODEL_HEADS[model_name]

    print(f"\n{'=' * 70}")
    print(f"Model: {model_name} ({n_layers} layers, {n_heads} heads)")
    print(f"{'=' * 70}")

    # Load TransformerLens model with fold_ln=True
    model = load_model_circuit_mode(model_name, device=device)

    # Build mean cache from ALL bands x draws data
    print("  Building mean activation cache...")
    all_input_ids = []
    all_target_ids = []
    for draw in DRAWS:
        for band in BANDS:
            ds = load_test_dataset(draw, band)
            all_input_ids.append(ds["input_ids"])
            all_target_ids.append(ds["target_ids"])

    cache_dataset = {
        "input_ids": np.concatenate(all_input_ids, axis=0),
        "target_ids": np.concatenate(all_target_ids, axis=0),
    }
    print(f"  Cache dataset: {len(cache_dataset['input_ids'])} examples")
    mean_cache = build_mean_cache(model, cache_dataset, batch_size=BATCH_SIZE)
    print(f"  Mean cache: {len(mean_cache)} hook points cached")

    # Extract names for residual stream
    resid_names = [f"blocks.{l}.hook_resid_post" for l in range(n_layers)]

    for draw in DRAWS:
        universal = universal_cache.get((model_name, draw))
        if universal is None:
            continue

        # Create ablation hooks for universal core
        ablation_hooks = create_ablation_hooks(universal, mean_cache, n_heads)
        print(f"\n  {draw}: {len(ablation_hooks)} ablation hooks")

        for test_band in BANDS:
            eval_idx += 1
            ds = load_test_dataset(draw, test_band)
            n_examples = ds["n_examples"]
            input_ids = ds["input_ids"]
            target_ids = ds["target_ids"]

            # Prepend BOS
            bos_id = model.tokenizer.bos_token_id
            input_ids_bos = np.concatenate(
                [np.full((n_examples, 1), bos_id, dtype=input_ids.dtype), input_ids],
                axis=1,
            )

            # Pre-allocate residual stream storage
            resid_post = np.zeros(
                (n_examples, n_layers, model.cfg.d_model), dtype=np.float32
            )

            # Register ablation hooks
            model.reset_hooks()
            for hook_name, hook_fn in ablation_hooks:
                model.add_hook(hook_name, hook_fn)

            try:
                n_batches = (n_examples + BATCH_SIZE - 1) // BATCH_SIZE
                with torch.no_grad():
                    for bi in range(n_batches):
                        s = bi * BATCH_SIZE
                        e = min(s + BATCH_SIZE, n_examples)
                        batch = torch.tensor(
                            input_ids_bos[s:e], dtype=torch.long, device=device
                        )

                        _, cache = model.run_with_cache(
                            batch,
                            names_filter=resid_names,
                            reset_hooks_end=False,
                            prepend_bos=False,
                        )

                        for l in range(n_layers):
                            resid_post[s:e, l] = (
                                cache[f"blocks.{l}.hook_resid_post"][:, PRED_POS, :]
                                .cpu()
                                .numpy()
                            )

                        del cache
                        torch.cuda.empty_cache()
            finally:
                model.reset_hooks()

            # Compute P(correct) at each layer
            prob_correct = compute_logit_lens_from_resid(model, resid_post, target_ids)
            # prob_correct: (n_examples, n_layers)

            # Store mean trajectory
            for l in range(n_layers):
                trajectory_rows.append(
                    {
                        "model": model_name,
                        "draw": draw,
                        "band": test_band,
                        "layer": l,
                        "mean_prob_correct": float(prob_correct[:, l].mean()),
                        "std_prob_correct": float(prob_correct[:, l].std()),
                        "median_prob_correct": float(np.median(prob_correct[:, l])),
                    }
                )

            final_p = prob_correct[:, -1].mean()
            print(
                f"    [{eval_idx}/{total_evals}] {test_band}: "
                f"final P(correct)={final_p:.4f}"
            )

            del resid_post, prob_correct
            torch.cuda.empty_cache()

    # Free model and mean cache
    del model, mean_cache
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nFreed {model_name} from GPU.")

df_traj = pd.DataFrame(trajectory_rows)
df_traj.to_csv(OUT_ANALYSIS / "universal_core_logit_trajectory.csv", index=False)
print(f"\nSaved: {OUT_ANALYSIS / 'universal_core_logit_trajectory.csv'}")
print(f"Total: {len(df_traj)} trajectory points")


Model: pythia-70m (6 layers, 8 heads)



Loading EleutherAI/pythia-70m (fold_ln=True)...


'torch_dtype' is deprecated! Use 'dtype' instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-70m into HookedTransformer
  Layers: 6, Heads: 8, d_model: 512, fold_ln: True
  Building mean activation cache...
  Cache dataset: 3375 examples


...

    [72/75] medium: final P(correct)=0.9870


    [73/75] high: final P(correct)=0.9773


    [74/75] very_high: final P(correct)=0.9682


    [75/75] control: final P(correct)=0.9723



Freed pythia-1.4b from GPU.

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/universal_core_logit_trajectory.csv
Total: 1230 trajectory points


## 4. Compute Convergence Layers

For each configuration, find the first layer where P(correct) >= 0.9 x final P(correct).

In [6]:
convergence_rows = []

for model_name in MODELS:
    n_layers = MODEL_LAYERS[model_name]

    for draw in DRAWS:
        for band in BANDS:
            sub = df_traj[
                (df_traj["model"] == model_name)
                & (df_traj["draw"] == draw)
                & (df_traj["band"] == band)
            ].sort_values("layer")

            if len(sub) == 0:
                continue

            probs = sub["mean_prob_correct"].values
            final_p = probs[-1]
            threshold = CONVERGENCE_THRESHOLD * final_p

            # Find first layer >= threshold
            conv_layer = n_layers - 1  # default: last layer
            for l in range(n_layers):
                if probs[l] >= threshold:
                    conv_layer = l
                    break

            convergence_rows.append(
                {
                    "model": model_name,
                    "draw": draw,
                    "band": band,
                    "convergence_layer": conv_layer,
                    "final_prob_correct": final_p,
                    "n_layers": n_layers,
                    "frac_convergence": conv_layer / n_layers,
                }
            )

df_conv = pd.DataFrame(convergence_rows)
df_conv.to_csv(OUT_ANALYSIS / "universal_core_convergence.csv", index=False)

print("Universal Core Convergence Layers:")
for model_name in MODELS:
    sub = df_conv[df_conv["model"] == model_name]
    print(
        f"  {model_name}: mean convergence layer = {sub['convergence_layer'].mean():.1f} "
        f"/ {sub['n_layers'].iloc[0]} layers"
    )

print(f"\nSaved: {OUT_ANALYSIS / 'universal_core_convergence.csv'}")

Universal Core Convergence Layers:
  pythia-70m: mean convergence layer = 5.0 / 6 layers
  pythia-160m: mean convergence layer = 11.0 / 12 layers
  pythia-410m: mean convergence layer = 21.2 / 24 layers
  pythia-1b: mean convergence layer = 12.5 / 16 layers
  pythia-1.4b: mean convergence layer = 20.7 / 24 layers

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/universal_core_convergence.csv


## 5. Trajectory Shape Comparison

Compare universal core trajectories with Phase 3 full circuit trajectories. Compute Pearson r (shape), convergence shift, and amplitude ratio.

In [7]:
comparison_rows = []

for model_name in MODELS:
    n_layers = MODEL_LAYERS[model_name]

    for draw in DRAWS:
        for band in BANDS:
            # Universal core trajectory
            uc_sub = df_traj[
                (df_traj["model"] == model_name)
                & (df_traj["draw"] == draw)
                & (df_traj["band"] == band)
            ].sort_values("layer")

            # Full circuit trajectory (Phase 3)
            fc_sub = df_circuit_traj[
                (df_circuit_traj["model"] == model_name)
                & (df_circuit_traj["draw"] == draw)
                & (df_circuit_traj["band"] == band)
            ].sort_values("layer")

            # Base model trajectory (Phase 3)
            base_sub = df_base_traj[
                (df_base_traj["model"] == model_name)
                & (df_base_traj["draw"] == draw)
                & (df_base_traj["band"] == band)
            ].sort_values("layer")

            if len(uc_sub) == 0 or len(fc_sub) == 0:
                continue

            uc_probs = uc_sub["mean_prob_correct"].values
            fc_probs = fc_sub["mean_prob_correct"].values

            # Ensure same number of layers
            n_common = min(len(uc_probs), len(fc_probs))
            uc_probs = uc_probs[:n_common]
            fc_probs = fc_probs[:n_common]

            # Pearson correlation (trajectory shape)
            if np.std(uc_probs) > 1e-10 and np.std(fc_probs) > 1e-10:
                r, p_val = sp_stats.pearsonr(uc_probs, fc_probs)
            else:
                r, p_val = np.nan, np.nan

            # Amplitude ratio
            amp_ratio = uc_probs[-1] / fc_probs[-1] if fc_probs[-1] > 1e-10 else np.nan

            # Convergence layer shift
            uc_conv = df_conv[
                (df_conv["model"] == model_name)
                & (df_conv["draw"] == draw)
                & (df_conv["band"] == band)
            ]
            fc_conv = df_circuit_conv[
                (df_circuit_conv["model"] == model_name)
                & (df_circuit_conv["draw"] == draw)
                & (df_circuit_conv["band"] == band)
            ]

            uc_conv_layer = (
                uc_conv.iloc[0]["convergence_layer"] if len(uc_conv) > 0 else np.nan
            )
            fc_conv_layer = (
                fc_conv.iloc[0]["mean_convergence_layer"]
                if len(fc_conv) > 0
                else np.nan
            )
            conv_shift = (
                uc_conv_layer - fc_conv_layer
                if not (np.isnan(uc_conv_layer) or np.isnan(fc_conv_layer))
                else np.nan
            )

            # Base model correlation (for reference)
            if len(base_sub) > 0:
                base_probs = base_sub["mean_prob_correct"].values[:n_common]
                if np.std(base_probs) > 1e-10 and np.std(uc_probs) > 1e-10:
                    r_base, _ = sp_stats.pearsonr(uc_probs, base_probs)
                else:
                    r_base = np.nan
            else:
                r_base = np.nan

            comparison_rows.append(
                {
                    "model": model_name,
                    "draw": draw,
                    "band": band,
                    "pearson_r_vs_full": r,
                    "pearson_p_vs_full": p_val,
                    "pearson_r_vs_base": r_base,
                    "amplitude_ratio": amp_ratio,
                    "uc_final_prob": uc_probs[-1],
                    "fc_final_prob": fc_probs[-1],
                    "uc_convergence_layer": uc_conv_layer,
                    "fc_convergence_layer": fc_conv_layer,
                    "convergence_shift": conv_shift,
                    "n_layers": n_common,
                }
            )

df_comparison = pd.DataFrame(comparison_rows)
df_comparison.to_csv(OUT_ANALYSIS / "algorithm_preservation.csv", index=False)

print("=" * 80)
print("ALGORITHM PRESERVATION ANALYSIS")
print("=" * 80)

for model_name in MODELS:
    sub = df_comparison[df_comparison["model"] == model_name]
    print(f"\n--- {model_name} ---")
    print(
        f"  Pearson r (vs full circuit):  mean={sub['pearson_r_vs_full'].mean():.4f}, "
        f"min={sub['pearson_r_vs_full'].min():.4f}"
    )
    print(f"  Pearson r (vs base model):   mean={sub['pearson_r_vs_base'].mean():.4f}")
    print(f"  Amplitude ratio:             mean={sub['amplitude_ratio'].mean():.4f}")
    print(
        f"  Convergence shift:           mean={sub['convergence_shift'].mean():.2f} layers"
    )

print(f"\nSaved: {OUT_ANALYSIS / 'algorithm_preservation.csv'}")

ALGORITHM PRESERVATION ANALYSIS

--- pythia-70m ---
  Pearson r (vs full circuit):  mean=0.7650, min=0.6354
  Pearson r (vs base model):   mean=0.7669
  Amplitude ratio:             mean=3.1475
  Convergence shift:           mean=0.49 layers

--- pythia-160m ---
  Pearson r (vs full circuit):  mean=0.9178, min=0.8735
  Pearson r (vs base model):   mean=0.9187
  Amplitude ratio:             mean=2.0443
  Convergence shift:           mean=1.98 layers

--- pythia-410m ---
  Pearson r (vs full circuit):  mean=0.9515, min=0.9301
  Pearson r (vs base model):   mean=0.9524
  Amplitude ratio:             mean=1.8108
  Convergence shift:           mean=2.21 layers

--- pythia-1b ---
  Pearson r (vs full circuit):  mean=0.9857, min=0.9747
  Pearson r (vs base model):   mean=0.9872
  Amplitude ratio:             mean=1.5688
  Convergence shift:           mean=1.02 layers

--- pythia-1.4b ---
  Pearson r (vs full circuit):  mean=0.9540, min=0.8801
  Pearson r (vs base model):   mean=0.9595
  Ampli

## 6. Visualizations

### VIZ T9_01: Trajectory Overlay (Base vs Full Circuit vs Universal Core)

In [8]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(20, 5), sharey=True)

for ax, model_name in zip(axes, MODELS):
    n_layers = MODEL_LAYERS[model_name]
    layers = np.arange(n_layers)

    # Average across draws and bands for each trajectory type
    # Base model
    base_sub = df_base_traj[df_base_traj["model"] == model_name]
    if len(base_sub) > 0:
        base_avg = base_sub.groupby("layer")["mean_prob_correct"].mean().values
        ax.plot(
            layers[: len(base_avg)],
            base_avg,
            color="gray",
            linewidth=2,
            alpha=0.7,
            label="Base model",
        )

    # Full circuit
    fc_sub = df_circuit_traj[df_circuit_traj["model"] == model_name]
    if len(fc_sub) > 0:
        fc_avg = fc_sub.groupby("layer")["mean_prob_correct"].mean().values
        ax.plot(
            layers[: len(fc_avg)],
            fc_avg,
            color="#1976D2",
            linewidth=2.5,
            label="Full circuit",
        )

    # Universal core
    uc_sub = df_traj[df_traj["model"] == model_name]
    if len(uc_sub) > 0:
        uc_avg = uc_sub.groupby("layer")["mean_prob_correct"].mean().values
        ax.plot(
            layers[: len(uc_avg)],
            uc_avg,
            color="#388E3C",
            linewidth=2.5,
            linestyle="--",
            label="Universal core",
        )

    ax.set_xlabel("Layer")
    ax.set_title(model_name, fontsize=11, fontweight="bold")
    ax.set_xticks(layers)

    # Add correlation annotation
    comp = df_comparison[df_comparison["model"] == model_name]
    if len(comp) > 0:
        mean_r = comp["pearson_r_vs_full"].mean()
        mean_amp = comp["amplitude_ratio"].mean()
        ax.text(
            0.02,
            0.98,
            f"r={mean_r:.3f}\namp={mean_amp:.2f}",
            transform=ax.transAxes,
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )

axes[0].set_ylabel("Mean P(correct)")
axes[0].legend(loc="center left", fontsize=9)
fig.suptitle(
    "P(correct) Trajectory: Base Model vs Full Circuit vs Universal Core\n"
    "(averaged across all draws and bands)",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T9_01_trajectory_overlay.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T9_01_trajectory_overlay.png


### VIZ T9_02: Convergence Layer Comparison

In [9]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(18, 4))

for ax, model_name in zip(axes, MODELS):
    # Build convergence data for 3 circuit types x bands
    circuit_types = ["Base", "Full Circuit", "Universal Core"]

    # Average convergence layer across draws for each band
    conv_data = np.full((3, len(BANDS)), np.nan)

    for j, band in enumerate(BANDS):
        # Base model - use Phase 3 base data if available
        # Base doesn't have convergence CSV, compute from trajectory
        base_sub = df_base_traj[
            (df_base_traj["model"] == model_name) & (df_base_traj["band"] == band)
        ]
        if len(base_sub) > 0:
            for draw in DRAWS:
                dsub = base_sub[base_sub["draw"] == draw].sort_values("layer")
                if len(dsub) > 0:
                    probs = dsub["mean_prob_correct"].values
                    final_p = probs[-1]
                    thresh = CONVERGENCE_THRESHOLD * final_p
                    for l in range(len(probs)):
                        if probs[l] >= thresh:
                            if np.isnan(conv_data[0, j]):
                                conv_data[0, j] = l
                            else:
                                conv_data[0, j] = (conv_data[0, j] + l) / 2
                            break

        # Full circuit
        fc_conv = df_circuit_conv[
            (df_circuit_conv["model"] == model_name) & (df_circuit_conv["band"] == band)
        ]
        if len(fc_conv) > 0:
            conv_data[1, j] = fc_conv["mean_convergence_layer"].mean()

        # Universal core
        uc_conv = df_conv[(df_conv["model"] == model_name) & (df_conv["band"] == band)]
        if len(uc_conv) > 0:
            conv_data[2, j] = uc_conv["convergence_layer"].mean()

    # Convert to DataFrame for seaborn
    conv_df = pd.DataFrame(conv_data, index=circuit_types, columns=BANDS)

    sns.heatmap(
        conv_df,
        annot=False,
        cmap="YlOrRd",
        ax=ax,
        square=True,
        linewidths=0,
        linecolor="none",
        cbar_kws={"shrink": 0.8, "label": "Layer"},
    )
    ax.set_xticklabels(BANDS, rotation=45, ha="right", fontsize=8)
    ax.set_title(model_name, fontsize=11, fontweight="bold")

fig.suptitle(
    "Convergence Layer: Base vs Full Circuit vs Universal Core\n"
    f"(first layer where P(correct) >= {CONVERGENCE_THRESHOLD} x final)",
    fontsize=12,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T9_02_convergence_comparison.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T9_02_convergence_comparison.png


### VIZ T9_03: Trajectory Correlation Scatter (Full Circuit vs Universal Core)

In [10]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(18, 5))

band_colors = {
    "low": "#2196F3",
    "medium": "#4CAF50",
    "high": "#FF9800",
    "very_high": "#F44336",
    "control": "#9E9E9E",
}

for ax, model_name in zip(axes, MODELS):
    n_layers = MODEL_LAYERS[model_name]

    # Collect all (full_circuit, universal_core) P(correct) pairs
    for band in BANDS:
        fc_vals = []
        uc_vals = []

        for draw in DRAWS:
            fc_sub = df_circuit_traj[
                (df_circuit_traj["model"] == model_name)
                & (df_circuit_traj["draw"] == draw)
                & (df_circuit_traj["band"] == band)
            ].sort_values("layer")
            uc_sub = df_traj[
                (df_traj["model"] == model_name)
                & (df_traj["draw"] == draw)
                & (df_traj["band"] == band)
            ].sort_values("layer")

            if len(fc_sub) > 0 and len(uc_sub) > 0:
                n = min(len(fc_sub), len(uc_sub))
                fc_vals.extend(fc_sub["mean_prob_correct"].values[:n].tolist())
                uc_vals.extend(uc_sub["mean_prob_correct"].values[:n].tolist())

        if fc_vals:
            ax.scatter(
                fc_vals,
                uc_vals,
                c=band_colors[band],
                alpha=0.5,
                s=20,
                label=band,
                edgecolors="none",
            )

    # Add identity line and best fit
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, lim], [0, lim], "k--", alpha=0.3, linewidth=1)

    # Fit overall line
    comp = df_comparison[df_comparison["model"] == model_name]
    mean_amp = comp["amplitude_ratio"].mean()
    ax.plot(
        [0, lim],
        [0, lim * mean_amp],
        color="green",
        linestyle="-",
        alpha=0.7,
        linewidth=1.5,
        label=f"slope={mean_amp:.2f}",
    )

    ax.set_xlabel("Full Circuit P(correct)")
    ax.set_title(model_name, fontsize=11, fontweight="bold")
    ax.set_aspect("equal")

axes[0].set_ylabel("Universal Core P(correct)")
axes[-1].legend(loc="upper left", fontsize=8, markerscale=1.5)
fig.suptitle(
    "Full Circuit vs Universal Core P(correct) per Layer\n"
    "Points on slope < 1 line = same algorithm, lower amplitude",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T9_03_trajectory_correlation.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T9_03_trajectory_correlation.png


## 7. Summary

In [11]:
print("=" * 80)
print("PHASE 5: INFORMATION FLOW: ALGORITHM PRESERVATION SUMMARY")
print("=" * 80)

print("\n--- Trajectory Correlations ---")
for model_name in MODELS:
    sub = df_comparison[df_comparison["model"] == model_name]
    r_mean = sub["pearson_r_vs_full"].mean()
    r_min = sub["pearson_r_vs_full"].min()
    r_base = sub["pearson_r_vs_base"].mean()
    print(f"  {model_name}:")
    print(f"    vs full circuit: r={r_mean:.4f} (min={r_min:.4f})")
    print(f"    vs base model:  r={r_base:.4f}")

print("\n--- Amplitude Ratios ---")
for model_name in MODELS:
    sub = df_comparison[df_comparison["model"] == model_name]
    amp = sub["amplitude_ratio"].mean()
    amp_std = sub["amplitude_ratio"].std()
    print(f"  {model_name}: {amp:.3f} +/- {amp_std:.3f}")

print("\n--- Convergence Shifts ---")
for model_name in MODELS:
    sub = df_comparison[df_comparison["model"] == model_name]
    shift = sub["convergence_shift"].mean()
    print(f"  {model_name}: {shift:+.2f} layers")

# Overall verdict
overall_r = df_comparison["pearson_r_vs_full"].mean()
overall_amp = df_comparison["amplitude_ratio"].mean()
overall_shift = df_comparison["convergence_shift"].mean()

print(f"\n--- VERDICT ---")
print(f"  Overall trajectory correlation: r = {overall_r:.4f}")
print(f"  Overall amplitude ratio:        {overall_amp:.3f}")
print(f"  Overall convergence shift:      {overall_shift:+.2f} layers")

if overall_r > 0.95:
    print(
        f"  ==> Universal core implements the SAME algorithm (r > 0.95), attenuated to {overall_amp:.0%}"
    )
elif overall_r > 0.85:
    print(
        f"  ==> Universal core implements a SIMILAR algorithm (r > 0.85), attenuated to {overall_amp:.0%}"
    )
else:
    print(f"  ==> Universal core implements a DIFFERENT algorithm (r < 0.85)")

print("\n--- Output Files ---")
for f in (
    sorted(OUT_ANALYSIS.glob("universal_core_logit*"))
    + sorted(OUT_ANALYSIS.glob("universal_core_conv*"))
    + sorted(OUT_ANALYSIS.glob("algorithm*"))
):
    print(f"  {f.name}")
for f in sorted(OUT_VIZ.glob("T9_*.png")):
    print(f"  {f.name}")

print("\nDone.")

PHASE 5: INFORMATION FLOW: ALGORITHM PRESERVATION SUMMARY

--- Trajectory Correlations ---
  pythia-70m:
    vs full circuit: r=0.7650 (min=0.6354)
    vs base model:  r=0.7669
  pythia-160m:
    vs full circuit: r=0.9178 (min=0.8735)
    vs base model:  r=0.9187
  pythia-410m:
    vs full circuit: r=0.9515 (min=0.9301)
    vs base model:  r=0.9524
  pythia-1b:
    vs full circuit: r=0.9857 (min=0.9747)
    vs base model:  r=0.9872
  pythia-1.4b:
    vs full circuit: r=0.9540 (min=0.8801)
    vs base model:  r=0.9595

--- Amplitude Ratios ---
  pythia-70m: 3.148 +/- 0.514
  pythia-160m: 2.044 +/- 0.227
  pythia-410m: 1.811 +/- 0.152
  pythia-1b: 1.569 +/- 0.116
  pythia-1.4b: 1.879 +/- 0.090

--- Convergence Shifts ---
  pythia-70m: +0.49 layers
  pythia-160m: +1.98 layers
  pythia-410m: +2.21 layers
  pythia-1b: +1.02 layers
  pythia-1.4b: +3.34 layers

--- VERDICT ---
  Overall trajectory correlation: r = 0.9148
  Overall amplitude ratio:        2.090
  Overall convergence shift:    